# 19 - Working with Online FMUs and the FMI Standard

A common question when evaluating multi-energy co-simulation tools is:
> **"Can we use online available FMUs and work with them in `energysim`?"**

The answer is **yes, absolutely!** `energysim` supports the open **Functional Mock-up Interface (FMI 2.0 & FMI 3.0)** standard for Co-Simulation (`csAdapter`). Any FMU compiled for your host architecture (Windows x64, Linux x64, macOS) can be downloaded directly from online repositories (such as the Modelica Association Cross-Check archive, GitHub Releases, or cloud storage) and seamlessly integrated into your simulation canvas.

In this tutorial, you will learn:
1. How to fetch and cache official FMI 2.0 Co-Simulation FMUs from online repositories.
2. How to introspect model descriptions, variables, and causality with `fmpy`.
3. How to co-simulate an online FMU inside `energysim.world`.
4. Best practices for step size selection and numerical stability.


In [ ]:
import os
import urllib.request
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fmpy

from energysim import world

print(f"fmpy version: {fmpy.__version__}")


## 1. Fetching an Online FMU

The [Modelica Association FMI Cross-Check Repository](https://github.com/modelica/fmi-cross-check) hosts official reference FMUs compiled for all major operating systems.

Here we define a helper function to download and cache an online FMU:


In [ ]:
CACHE_DIR = Path("fmu_cache")
CACHE_DIR.mkdir(exist_ok=True)

# URL to official Modelica Cross-Check FMI 2.0 Co-Simulation FMU
FMU_URL = (
    "https://raw.githubusercontent.com/modelica/fmi-cross-check/master/"
    "fmus/2.0/cs/win64/Test-FMUs/0.0.2/BouncingBall/BouncingBall.fmu"
)
FMU_PATH = CACHE_DIR / "BouncingBall.fmu"

if not FMU_PATH.exists():
    print(f"Downloading online FMU from {FMU_URL}...")
    req = urllib.request.Request(FMU_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(FMU_PATH, "wb") as f:
        f.write(resp.read())
    print(f"Downloaded successfully! File size: {FMU_PATH.stat().st_size:,} bytes")
else:
    print(f"Using cached FMU: {FMU_PATH} ({FMU_PATH.stat().st_size:,} bytes)")


## 2. Introspecting Model Variables & Metadata

Before adding an FMU to an `energysim` world, we can inspect its model description to see available variables, their causality (`input`, `output`, `parameter`, `local`), and units:


In [ ]:
md = fmpy.read_model_description(str(FMU_PATH), validate=False)

print(f"Model Name  : {md.modelName}")
print(f"FMI Version : {md.fmiVersion}")
print(f"Platforms   : {fmpy.supported_platforms(str(FMU_PATH))}")
print(f"Total Vars  : {len(md.modelVariables)}")

# Display variables in a clean table
var_data = []
for v in md.modelVariables:
    var_data.append({
        "Name": v.name,
        "Causality": v.causality,
        "Type": v.type,
        "Start": getattr(v, "start", None),
        "Description": v.description
    })
df_vars = pd.DataFrame(var_data)
display(df_vars)


## 3. Co-Simulating the FMU in `energysim.world`

Now we plug the online FMU into an `energysim.world` canvas.
We record output trajectories for position `h` (height) and velocity `v`, using a micro-step of 0.01 s and macro exchange step of 0.05 s:


In [ ]:
sim = world(start_time=0.0, stop_time=3.0, t_macro=0.05)

# Add the downloaded FMU
sim.add_fmu(
    sim_name="ball",
    model=str(FMU_PATH),
    step_size=0.01,
    outputs=["h", "v"],
    validate=False
)

# Simulate
res = sim.simulate(pbar=False)
df_res = res.to_frame()

print(f"Simulation completed! Recorded {len(df_res)} time steps.")
display(df_res.head())


## 4. Visualizing Results

Let's plot the continuous kinematic trajectory and bouncing impacts:


In [ ]:
time_s = df_res.index

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)

ax1.plot(time_s, df_res["ball.h"], color="royalblue", lw=2, label="Height h(t)")
ax1.axhline(0, color="gray", ls="--", lw=0.8)
ax1.set_ylabel("Height [m]")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title("Online FMI 2.0 Co-Simulation in energysim (BouncingBall.fmu)")

ax2.plot(time_s, df_res["ball.v"], color="crimson", lw=1.8, label="Velocity v(t)")
ax2.axhline(0, color="gray", ls="--", lw=0.8)
ax2.set_ylabel("Velocity [m/s]")
ax2.set_xlabel("Time [seconds]")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Key Best Practices for FMUs in `energysim`

1. **Host Architecture**: Ensure the online FMU contains binaries for your target OS (`win64`, `linux64`, or `darwin64`). Check with `fmpy.supported_platforms(fmu_path)`.
2. **Step Size ($t_{\text{micro}}$)**: Choose an FMU `step_size` smaller than the smallest internal time constant of the physical model. If an internal filter has $T = 5\text{ s}$, a step size of $1\text{ s}$ avoids numerical instability.
3. **Unit Consistency**: Declare `units={...}` in `sim.add_fmu(...)` if the FMU's internal XML lacks standard unit attributes. `energysim` handles unit conversion automatically across simulators!
